# Milestone 2: Build Vector & Graph RAG Indices
This notebook acts as the data preparation step, generating the ChromaDB Vector Index and the NetworkX Graph Index on the local Kaggle disk (`/kaggle/working/`) using the ContractNLI training split.

In [ ]:
!pip install -q chromadb sentence-transformers networkx

In [ ]:
import json
import os
import chromadb
from sentence_transformers import SentenceTransformer

# 1. Setup local storage in the Kaggle working directory
# This stays on the Kaggle instance disk
client = chromadb.PersistentClient(path="/kaggle/working/vector_db")
collection = client.get_or_create_collection(name="m2_training_rag")

# 2. Use a lightweight embedding model (runs on Kaggle CPU/GPU)
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# 3. Load the Training Split
train_path = "/kaggle/working/contract-nli/train.json"
if not os.path.exists(train_path):
    train_path = "sample.json" # Fallback for local testing

with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)

print(f"Loaded {len(train_data.get('documents', []))} documents from {train_path}")

### Structured Indexing
Instead of chunking the raw legal text (`text_content[i:i+600]`), which introduces massive noise, we use our **Data Strategy**: extracting the labeled spans paired with their hypothesis and entailed/contradicted outcomes acting as a precedent knowledge base.

In [ ]:
labels_map = train_data.get('labels', {})
docs_to_index = []
metadatas = []
ids = []
chunk_idx = 0

for doc in train_data.get('documents', []):
    doc_id = str(doc['id'])
    text = doc['text']
    doc_spans = doc.get('spans', [])
    annotations = doc.get('annotation_sets', [{}])[0].get('annotations', {})
    
    for hyp_id, ann in annotations.items():
        choice = ann.get('choice')
        span_indices = ann.get('spans', [])
        hypothesis_text = labels_map.get(hyp_id, {}).get('hypothesis', '')
        
        if choice in ['Entailment', 'Contradiction'] and span_indices:
            for idx in span_indices:
                if idx >= len(doc_spans): continue
                span_val = doc_spans[idx]
                
                if isinstance(span_val, list) and len(span_val) == 2:
                    ev_text = text[span_val[0]:span_val[1]]
                elif isinstance(span_val, str):
                    ev_text = span_val
                else:
                    continue
                    
                # Build the structured context block
                chunk_text = (
                    f"Hypothesis: '{hypothesis_text}'\n"
                    f"Historical Outcome: {choice}\n"
                    f"Supporting Clause: '{ev_text}'"
                )
                
                docs_to_index.append(chunk_text)
                metadatas.append({"source_doc": doc_id, "hypothesis": hyp_id, "outcome": choice})
                ids.append(f"{doc_id}_{chunk_idx}")
                chunk_idx += 1

if docs_to_index:
    print(f"Embedding and inserting {len(docs_to_index)} structured items into ChromaDB...")
    embeddings = embed_model.encode(docs_to_index).tolist()
    
    collection.add(
        documents=docs_to_index,
        embeddings=embeddings,
        ids=ids,
        metadatas=metadatas
    )
    print("Vector RAG indexed and stored in /kaggle/working/vector_db")
else:
    print("No data found to index. Check the dataset path.")